# 04 — Sensor Calibration

> **참고:** CameraLidarCalib 포트폴리오 프로젝트
> **언어:** Python (OpenCV, Open3D, SciPy)
> **목표:** Camera 내부 파라미터 + Camera-LiDAR 외부 파라미터 추정

---

## 4-1. 좌표계 체인



```
LiDAR 좌표계  →  [R_L2C | t_L2C]  →  Camera 좌표계  →  K  →  이미지
     Pₗ                                    Pc = R·Pₗ + t          u,v
```




**전체 투영:**


```
[u]       [X_c]       [R | t]   [X_l]
[v]  = K · proj(Pc),  Pc = ──── · [Y_l]
[1]                            [Z_l]
                                [ 1 ]
```




---

## 4-2. 카메라 내부 파라미터 보정 (Intrinsic Calibration)

### 체커보드 패턴 사용



In [ ]:
import cv2
import numpy as np
import glob

BOARD_W, BOARD_H = 9, 6       # 내부 코너 수
SQUARE_SIZE = 0.025            # 정사각형 크기 [m]

objp = np.zeros((BOARD_H * BOARD_W, 3), np.float32)
objp[:, :2] = np.mgrid[0:BOARD_W, 0:BOARD_H].T.reshape(-1, 2) * SQUARE_SIZE

obj_points = []   # 3D 실세계 점
img_points = []   # 2D 이미지 점

images = glob.glob("calib_images/*.png")
for fname in images:
    img = cv2.imread(fname, cv2.IMREAD_GRAYSCALE)
    ret, corners = cv2.findChessboardCorners(img, (BOARD_W, BOARD_H), None)
    if ret:
        # 서브픽셀 정밀화
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
        corners2 = cv2.cornerSubPix(img, corners, (11, 11), (-1, -1), criteria)
        obj_points.append(objp)
        img_points.append(corners2)

# 보정 실행
ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    obj_points, img_points, img.shape[::-1], None, None
)
print(f"RMS reprojection error: {ret:.4f} px")
print(f"K =\n{K}")
# 좋은 결과: < 0.5 px (정밀 렌즈: 0.035 px)




---

## 4-3. LiDAR 평면 추출 (RANSAC)

체커보드 평면을 LiDAR 포인트 클라우드에서 추출.



In [ ]:
import open3d as o3d

def extract_checkerboard_plane(pcd: o3d.geometry.PointCloud,
                                dist_thresh=0.01):
    """RANSAC으로 체커보드 평면 추출 → 평면 법선 + 인라이어"""
    plane_model, inliers = pcd.segment_plane(
        distance_threshold=dist_thresh,
        ransac_n=3,
        num_iterations=1000
    )
    # plane_model = [a, b, c, d]  (ax + by + cz + d = 0)
    inlier_cloud = pcd.select_by_index(inliers)
    return plane_model, inlier_cloud

# 포인트 클라우드 로딩
pcd = o3d.io.read_point_cloud("scene_001.pcd")

# ROI 제한 (체커보드 근처)
bbox = o3d.geometry.AxisAlignedBoundingBox(
    min_bound=[-1, -1, 0.5],
    max_bound=[ 1,  1, 5.0]
)
pcd_roi = pcd.crop(bbox)

plane_model, board_cloud = extract_checkerboard_plane(pcd_roi)
print(f"Plane: {plane_model}")
print(f"Inliers: {len(board_cloud.points)} points")




---

## 4-4. 체커보드 코너 3D 위치 추정



In [ ]:
def estimate_3d_corners_from_lidar(board_cloud, board_w=9, board_h=6,
                                    square_size=0.025):
    """
    LiDAR 평면 내에서 체커보드 코너 3D 위치 추정
    → 평면 PCA → 2D 그리드 피팅 → 3D 복원
    """
    pts = np.asarray(board_cloud.points)
    # PCA로 평면 좌표계
    centroid = pts.mean(axis=0)
    centered = pts - centroid
    _, _, Vt = np.linalg.svd(centered)
    # Vt[0], Vt[1]: 평면 내 축, Vt[2]: 법선
    u_axis = Vt[0]
    v_axis = Vt[1]

    # 2D 투영 → 그리드 정규화
    pts_2d = np.stack([centered @ u_axis, centered @ v_axis], axis=1)

    # 전체 코너 위치 (이상적인 그리드)
    corners_3d = []
    for row in range(board_h):
        for col in range(board_w):
            p2d = np.array([col * square_size, row * square_size])
            # 2D → 3D 복원
            p3d = centroid + p2d[0] * u_axis + p2d[1] * v_axis
            corners_3d.append(p3d)
    return np.array(corners_3d)




---

## 4-5. 외부 파라미터 추정 (PnP)

카메라 코너 2D ↔ LiDAR 코너 3D 매칭 → PnP로 R, t 추정.



In [ ]:
def estimate_extrinsic(corners_3d: np.ndarray,  # (N, 3) LiDAR 좌표
                        corners_2d: np.ndarray,  # (N, 2) 이미지 픽셀
                        K: np.ndarray,
                        dist: np.ndarray):
    """
    solvePnP: 3D-2D 대응점 → R, t (LiDAR→Camera)
    """
    success, rvec, tvec, inliers = cv2.solvePnPRansac(
        corners_3d.astype(np.float64),
        corners_2d.astype(np.float64),
        K, dist,
        iterationsCount=1000,
        reprojectionError=2.0,
        confidence=0.999
    )
    if not success:
        raise RuntimeError("PnP 실패")
    R, _ = cv2.Rodrigues(rvec)
    return R, tvec.flatten(), inliers

# 여러 장면에서 추정 → 평균
Rs, ts = [], []
for R_i, t_i in scene_estimates:
    Rs.append(R_i)
    ts.append(t_i)

# Rotation 평균 (Geodesic 평균)
from scipy.spatial.transform import Rotation
rots = Rotation.from_matrix(Rs)
R_mean = rots.mean().as_matrix()
t_mean = np.mean(ts, axis=0)




---

## 4-6. Bundle Adjustment (정밀화)



In [ ]:
from scipy.optimize import least_squares

def pack_params(R: np.ndarray, t: np.ndarray) -> np.ndarray:
    rvec, _ = cv2.Rodrigues(R)
    return np.concatenate([rvec.flatten(), t.flatten()])

def unpack_params(params: np.ndarray):
    rvec = params[:3].reshape(3, 1)
    t    = params[3:6]
    R, _ = cv2.Rodrigues(rvec)
    return R, t

def reprojection_residuals(params, pts3d_list, pts2d_list, K, dist):
    R, t = unpack_params(params)
    residuals = []
    for pts3d, pts2d in zip(pts3d_list, pts2d_list):
        proj, _ = cv2.projectPoints(pts3d, cv2.Rodrigues(R)[0], t.reshape(3, 1), K, dist)
        proj = proj.reshape(-1, 2)
        residuals.append((proj - pts2d).flatten())
    return np.concatenate(residuals)

x0 = pack_params(R_mean, t_mean)
result = least_squares(
    reprojection_residuals, x0,
    args=(pts3d_all, pts2d_all, K, dist),
    method='lm', max_nfev=2000
)
R_opt, t_opt = unpack_params(result.x)
print(f"Final RMSE: {np.sqrt(result.cost / len(result.fun)):.4f} px")




---

## 4-7. 결과 검증 — LiDAR 점 투영



In [ ]:
def project_lidar_to_image(pts_lidar: np.ndarray,   # (N, 3)
                            R: np.ndarray, t: np.ndarray,
                            K: np.ndarray,
                            img_shape: tuple) -> tuple:
    """LiDAR 포인트 → 이미지 픽셀, 깊이 컬러 시각화"""
    # 카메라 좌표
    pts_cam = (R @ pts_lidar.T).T + t   # (N, 3)
    # 카메라 앞쪽만 (Z > 0)
    mask = pts_cam[:, 2] > 0
    pts_cam = pts_cam[mask]

    # 이미지 투영
    uvw = (K @ pts_cam.T).T
    uv = (uvw[:, :2] / uvw[:, 2:3]).astype(int)

    # 이미지 범위 내 필터
    H, W = img_shape
    in_fov = (uv[:, 0] >= 0) & (uv[:, 0] < W) & \
             (uv[:, 1] >= 0) & (uv[:, 1] < H)
    uv = uv[in_fov]
    depth = pts_cam[in_fov, 2]
    return uv, depth

# 시각화
import cv2
import matplotlib.pyplot as plt

img = cv2.imread("frame.png")
uv, depth = project_lidar_to_image(pts_lidar, R_opt, t_opt, K, img.shape[:2])

# depth → 컬러 (파랑=가까움, 빨강=멀음)
depth_norm = (depth - depth.min()) / (depth.max() - depth.min())
colors = plt.cm.jet(depth_norm)[:, :3] * 255  # (N, 3) RGB

img_vis = img.copy()
for (u, v), color in zip(uv, colors):
    cv2.circle(img_vis, (u, v), 2, color.tolist(), -1)
cv2.imwrite("projection_overlay.png", img_vis)




---

## 4-8. 핵심 개념 정리

| 개념 | 핵심 |
|------|------|
| 내부 파라미터 (K) | fx, fy, cx, cy + 왜곡 계수 |
| 외부 파라미터 (R, t) | LiDAR → Camera 좌표 변환 |
| PnP (Perspective-n-Point) | 3D-2D 대응 → 카메라 포즈 추정 |
| RANSAC | 대응점 오류(outlier)에 강건한 추정 |
| Bundle Adjustment | 비선형 최적화로 재투영 오차 최소화 |
| 재투영 오차 | 정확도 지표: 목표 < 1.0 px |

---

## 다음 단계

→ **05_kalman_filter.md** : Bayes 필터 → 선형 KF → EKF → UKF
